# DrugBank EDA: Complete XML and CSV Analysis

Comprehensive analysis of DrugBank XML database and Link files with df.head() displays.

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import xml.etree.ElementTree as ET
from IPython.display import display
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
plt.style.use('bmh')
db_path = r'D\ADE DATASET DOWNLOAD\drugbank'
eda_path = r'D:\ADE DATASET DOWNLOAD\EDA'

## 1. Drug Links CSV

In [ ]:
drug_links = pd.read_csv(os.path.join(db_path, 'drug links.csv'))print(f"Total Drug Link Records: {len(drug_links):,}")display(drug_links.head(5))display(drug_links.info())coverage = drug_links.notnull().mean() * 100fig, axes = plt.subplots(2, 2, figsize=(18, 12))# Coverage bar chartcoverage.sort_values(ascending=True).plot(kind='barh', ax=axes[0,0], color='teal')axes[0,0].set_title('External ID Coverage for Drug Links', fontsize=14, fontweight='bold')axes[0,0].set_xlabel('Coverage %')axes[0,0].axvline(50, color='red', linestyle='--', alpha=0.5, label='50% threshold')axes[0,0].legend()# Number of external IDs per drugexternal_id_count = drug_links.notnull().sum(axis=1) - 2  # Exclude DrugBank ID and Namesns.histplot(external_id_count, bins=20, kde=True, ax=axes[0,1], color='purple')axes[0,1].set_title('External IDs per Drug', fontsize=14, fontweight='bold')axes[0,1].set_xlabel('Number of External IDs')axes[0,1].axvline(external_id_count.median(), color='red', linestyle='--',                  label=f'Median: {external_id_count.median():.0f}')axes[0,1].legend()# Coverage heatmap (top fields)top_fields = coverage.sort_values(ascending=False).head(15)sns.barplot(x=top_fields.values, y=top_fields.index, ax=axes[1,0], palette='coolwarm')axes[1,0].set_title('Top 15 Database Coverage', fontsize=14, fontweight='bold')axes[1,0].set_xlabel('Coverage %')# Summary tablesummary_stats = pd.DataFrame({    'Total Drugs': [len(drug_links)],    'Total Fields': [len(drug_links.columns)],    'Avg Coverage': [coverage.mean()],    'Fully Linked': [(external_id_count == len(drug_links.columns) - 2).sum()]})axes[1,1].axis('off')table = axes[1,1].table(cellText=summary_stats.T.values,                        rowLabels=summary_stats.T.index,                       colLabels=['Value'],                       cellLoc='center', loc='center')table.auto_set_font_size(False)table.set_fontsize(11)table.scale(1, 3)axes[1,1].set_title('Drug Links Summary', fontsize=14, fontweight='bold')plt.tight_layout()plt.show()print(f"\nCoverage Statistics:")print(f"  Highest coverage: {coverage.idxmax()} ({coverage.max():.1f}%)")print(f"  Lowest coverage: {coverage.idxmin()} ({coverage.min():.1f}%)")print(f"  Average coverage: {coverage.mean():.1f}%")

## 2. Structure Links CSV

In [ ]:
structure_links = pd.read_csv(os.path.join(db_path, 'structure links.csv'))print(f"Total Structure Link Records: {len(structure_links):,}")display(structure_links.head(5))if not structure_links.empty:    fig, axes = plt.subplots(2, 2, figsize=(18, 12))        # Structure sources distribution    if 'Source' in structure_links.columns:        source_dist = structure_links['Source'].value_counts()        axes[0,0].pie(source_dist.values, labels=source_dist.index, autopct='%1.1f%%',                      startangle=90, colors=sns.color_palette('Set3'))        axes[0,0].set_title('Structure Source Distribution', fontsize=14, fontweight='bold')        # Structures per drug    struct_per_drug = structure_links.groupby('DrugBank ID').size()    sns.histplot(struct_per_drug, bins=20, kde=True, ax=axes[0,1], color='darkblue')    axes[0,1].set_title('Structure Links per Drug', fontsize=14, fontweight='bold')    axes[0,1].set_xlabel('Number of Structure Links')    axes[0,1].axvline(struct_per_drug.median(), color='red', linestyle='--',                     label=f'Median: {struct_per_drug.median():.0f}')    axes[0,1].legend()        # Coverage analysis    coverage = structure_links.notnull().mean() * 100    coverage.sort_values(ascending=True).plot(kind='barh', ax=axes[1,0], color='coral')    axes[1,0].set_title('Field Coverage in Structure Links', fontsize=14, fontweight='bold')    axes[1,0].set_xlabel('Coverage %')        # Summary stats    summary = pd.DataFrame({        'Total Links': [len(structure_links)],        'Unique Drugs': [structure_links['DrugBank ID'].nunique()],        'Avg Links/Drug': [struct_per_drug.mean()],        'Max Links/Drug': [struct_per_drug.max()]    })    axes[1,1].axis('off')    table = axes[1,1].table(cellText=summary.T.values,                            rowLabels=summary.T.index,                           colLabels=['Value'],                           cellLoc='center', loc='center')    table.auto_set_font_size(False)    table.set_fontsize(11)    table.scale(1, 3)    axes[1,1].set_title('Structure Links Summary', fontsize=14, fontweight='bold')        plt.tight_layout()    plt.show()        print(f"\nStructure Statistics:")    print(f"  Unique drugs with structures: {structure_links['DrugBank ID'].nunique():,}")    print(f"  Average structures per drug: {struct_per_drug.mean():.2f}")

## 3. UniProt Links CSV

In [ ]:
uniprot_links = pd.read_csv(os.path.join(db_path, 'uniprot links.csv'))print(f"Total UniProt Link Records: {len(uniprot_links):,}")display(uniprot_links.head(5))promiscuity = uniprot_links.groupby('DrugBank ID').size().reset_index(name='target_count')top_targets = uniprot_links['UniProt Name'].value_counts().head(20)fig, axes = plt.subplots(2, 2, figsize=(18, 12))# Polypharmacology distributionsns.histplot(promiscuity['target_count'], bins=30, kde=True, ax=axes[0,0], color='purple')axes[0,0].set_title('Polypharmacology: Targets per Drug', fontsize=14, fontweight='bold')axes[0,0].set_xlabel('Number of Targets')axes[0,0].axvline(promiscuity['target_count'].median(), color='red', linestyle='--',                 label=f'Median: {promiscuity["target_count"].median():.0f}')axes[0,0].legend()# Top targetssns.barplot(x=top_targets.values, y=top_targets.index, ax=axes[0,1], palette='rocket')axes[0,1].set_title('Top 20 Most Frequently Targeted Proteins', fontsize=14, fontweight='bold')axes[0,1].set_xlabel('Number of Drugs')# Promiscuity categoriespromiscuity_cat = pd.cut(promiscuity['target_count'],                          bins=[0, 1, 3, 5, 10, float('inf')],                         labels=['1 target', '2-3 targets', '4-5 targets', '6-10 targets', '10+ targets'])cat_counts = promiscuity_cat.value_counts().sort_index()axes[1,0].pie(cat_counts.values, labels=cat_counts.index, autopct='%1.1f%%',              startangle=90, colors=sns.color_palette('Set2'))axes[1,0].set_title('Drug Promiscuity Categories', fontsize=14, fontweight='bold')# Drugs per targetdrugs_per_target = uniprot_links.groupby('UniProt Name').size()sns.violinplot(y=drugs_per_target, ax=axes[1,1], color='gold')axes[1,1].set_title('Drugs Targeting Each Protein', fontsize=14, fontweight='bold')axes[1,1].set_ylabel('Number of Drugs')plt.tight_layout()plt.show()print(f"\nTarget-Drug Statistics:")print(f"  Total unique drugs: {uniprot_links['DrugBank ID'].nunique():,}")print(f"  Total unique targets: {uniprot_links['UniProt Name'].nunique():,}")print(f"  Average targets per drug: {promiscuity['target_count'].mean():.2f}")print(f"  Average drugs per target: {drugs_per_target.mean():.2f}")print(f"  Multi-target drugs (3+ targets): {(promiscuity['target_count'] >= 3).sum():,} ({(promiscuity['target_count'] >= 3).sum()/len(promiscuity)*100:.1f}%)")

## 4. XML Data Extraction (Sampling)

In [ ]:
def extract_drugbank_features(xml_path, limit=2000):
    ns = '{http://www.drugbank.ca}'
    drugs = []
    context = ET.iterparse(xml_path, events=('end',))
    count = 0
    
    for event, elem in context:
        if elem.tag == f"{ns}drug":
            db_id = elem.findtext(f"{ns}drugbank-id[@primary='true']")
            name = elem.findtext(f"{ns}name")
            type_ = elem.get('type')
            
            interactions = elem.find(f"{ns}drug-interactions")
            ddi_count = len(interactions.findall(f"{ns}drug-interaction")) if interactions is not None else 0
            
            groups = [g.text for g in elem.findall(f"{ns}groups/{ns}group")]
            categories = [c.findtext(f"{ns}category") for c in elem.findall(f"{ns}categories/{ns}category")]
            
            drugs.append({
                'drugbank_id': db_id,
                'name': name,
                'type': type_,
                'ddi_count': ddi_count,
                'is_approved': 'approved' in groups,
                'main_category': categories[0] if categories else 'Unknown'
            })
            
            elem.clear()
            count += 1
            if limit and count >= limit: break
            
    return pd.DataFrame(drugs)

xml_file = os.path.join(db_path, 'full database.xml')
if os.path.exists(xml_file):
    print("Extracting drug features from XML (this may take a few minutes)...")
    db_features = extract_drugbank_features(xml_file, limit=2000)
    print(f"Extracted features for {len(db_features)} drugs")
    display(db_features.head(5))
else:
    print("XML file not found")
    db_features = pd.DataFrame()

## 5. Therapeutic Categories

In [ ]:
if not db_features.empty and 'main_category' in db_features.columns:    top_cats = db_features['main_category'].value_counts().head(20)        fig, axes = plt.subplots(2, 2, figsize=(18, 12))        # Top categories bar chart    sns.barplot(x=top_cats.values, y=top_cats.index, ax=axes[0,0], palette='viridis')    axes[0,0].set_title('Top 20 Therapeutic Categories', fontsize=14, fontweight='bold')    axes[0,0].set_xlabel('Number of Drugs')        # Pie chart for top 10    top10 = db_features['main_category'].value_counts().head(10)    axes[0,1].pie(top10.values, labels=top10.index, autopct='%1.1f%%',                  startangle=90, colors=sns.color_palette('tab20'))    axes[0,1].set_title('Top 10 Categories Proportion', fontsize=14, fontweight='bold')        # Drug type distribution    if 'type' in db_features.columns:        type_dist = db_features['type'].value_counts()        sns.barplot(x=type_dist.values, y=type_dist.index, ax=axes[1,0], palette='Set1')        axes[1,0].set_title('Drug Type Distribution', fontsize=14, fontweight='bold')        axes[1,0].set_xlabel('Count')        # Approval status    if 'is_approved' in db_features.columns:        approval_counts = db_features['is_approved'].value_counts()        axes[1,1].pie(approval_counts.values,                      labels=['Approved' if x else 'Not Approved' for x in approval_counts.index],                     autopct='%1.1f%%', startangle=90, colors=['green', 'red'])        axes[1,1].set_title('Drug Approval Status', fontsize=14, fontweight='bold')        plt.tight_layout()    plt.show()        print(f"\nTherapeutic Category Statistics:")    print(f"  Total unique categories: {db_features['main_category'].nunique():,}")    if 'is_approved' in db_features.columns:        print(f"  Approved drugs: {db_features['is_approved'].sum():,} ({db_features['is_approved'].sum()/len(db_features)*100:.1f}%)")

## 6. Drug-Drug Interaction Analysis

In [ ]:
if not db_features.empty and 'ddi_count' in db_features.columns:    fig, axes = plt.subplots(2, 2, figsize=(18, 12))        # Boxplot by drug type    sns.boxplot(data=db_features[db_features['ddi_count'] > 0], x='type', y='ddi_count',                ax=axes[0,0], palette='Set2')    axes[0,0].set_yscale('log')    axes[0,0].set_title('Interaction Burden by Drug Type (Log Scale)', fontsize=14, fontweight='bold')    axes[0,0].set_xticklabels(axes[0,0].get_xticklabels(), rotation=45, ha='right')        # DDI distribution histogram    ddi_with_interactions = db_features[db_features['ddi_count'] > 0]['ddi_count']    sns.histplot(ddi_with_interactions, bins=40, kde=True, ax=axes[0,1], color='darkred')    axes[0,1].set_title('DDI Count Distribution (Drugs with DDIs)', fontsize=14, fontweight='bold')    axes[0,1].set_xlabel('Number of Drug-Drug Interactions')    axes[0,1].axvline(ddi_with_interactions.median(), color='yellow', linestyle='--',                     label=f'Median: {ddi_with_interactions.median():.0f}')    axes[0,1].legend()        # DDI burden categories    ddi_categories = pd.cut(db_features['ddi_count'],                            bins=[0, 1, 10, 50, 100, float('inf')],                           labels=['No DDI', '1-10', '11-50', '51-100', '100+'])    cat_counts = ddi_categories.value_counts().sort_index()    sns.barplot(x=cat_counts.index, y=cat_counts.values, ax=axes[1,0], palette='Reds')    axes[1,0].set_title('DDI Burden Categories', fontsize=14, fontweight='bold')    axes[1,0].set_xlabel('DDI Range')    axes[1,0].set_ylabel('Number of Drugs')    axes[1,0].set_xticklabels(axes[1,0].get_xticklabels(), rotation=45)        # Top 15 highest DDI drugs    top_ddi = db_features.sort_values('ddi_count', ascending=False).head(15)    sns.barplot(data=top_ddi, x='ddi_count', y='name', ax=axes[1,1], palette='magma')    axes[1,1].set_title('Top 15 Drugs by DDI Count', fontsize=14, fontweight='bold')    axes[1,1].set_xlabel('Number of Interactions')        plt.tight_layout()    plt.show()        print(f"\nDDI Statistics:")    print(f"  Drugs with DDI data: {(db_features['ddi_count'] > 0).sum():,} ({(db_features['ddi_count'] > 0).sum()/len(db_features)*100:.1f}%)")    print(f"  Average DDIs per drug (with DDIs): {ddi_with_interactions.mean():.2f}")    print(f"  Max DDIs for one drug: {db_features['ddi_count'].max()}")        print("\nDrugs with Highest DDI Profile:")    display(top_ddi[['name', 'main_category', 'ddi_count']].head(10))

## 7. Summary Report

In [ ]:
print("\n" + "="*60)
print("DRUGBANK ANALYSIS SUMMARY")
print("="*60)
print(f"Drug links analyzed: {len(drug_links):,}")
print(f"Structure links analyzed: {len(structure_links):,}")
print(f"UniProt links analyzed: {len(uniprot_links):,}")
print(f"XML drugs extracted: {len(db_features):,}")
print("="*60)